In [1]:
import sys
sys.path.append("..")
from microfluidics_controllers import (TestMicrofluidicsController, TestValveController,
TestPipettePump, MicrofluidicsControllerWidget)

# Create a test microfluidics controller
Here we demonstrate how to connect and use an object of the MicrofluidicsController class.
We use the TestMicrofluidicsControllers since this does not require any hardware to test with.

In [2]:
ctrl = TestMicrofluidicsController()

# 1. Connect to the controller
print("Is connected?", ctrl.connected)
ctrl.connect()
print("Is connected?", ctrl.connected)

# 2. Check number of channels
print("Number of channels:", ctrl.get_number_channels())


print("Channel 1 pressure:", ctrl.get_pressure(1))
print("Channel 2 pressure:", ctrl.get_pressure(2))
print("Channel 3 pressure:", ctrl.get_pressure(3))

# 3. Set pressures on all channels
ctrl.verbose = True
ctrl.set_pressure(1, 500)
ctrl.set_pressure(2, 1000)
ctrl.set_pressure(3, 1500)

# 4. Get pressures back
print("Channel 1 pressure:", ctrl.get_pressure(1))
print("Channel 2 pressure:", ctrl.get_pressure(2))
print("Channel 3 pressure:", ctrl.get_pressure(3))

# 5. Change one channel
ctrl.set_pressure(2, 750)
print("Updated channel 2 pressure:", ctrl.get_pressure(2))

# 6. Disconnect
ctrl.disconnect()
print("Is connected?", ctrl.connected)

Is connected? False
Connected to test pump controller
Is connected? True
Number of channels: 3
Channel 1 pressure: 0
Channel 2 pressure: 0
Channel 3 pressure: 0
Set pressure of channel 1 to 500 mbar
Set pressure of channel 2 to 1000 mbar
Set pressure of channel 3 to 1500 mbar
Channel 1 pressure: 500
Channel 2 pressure: 1000
Channel 3 pressure: 1500
Set pressure of channel 2 to 750 mbar
Updated channel 2 pressure: 750
Disconnected from test pump controller
Is connected? False


# Create a test pipette pump

In [3]:
pump = TestPipettePump()

# 1. Connect to the pump
pump.connect()

# 2. Check connection status
print("Is connected?", pump.is_connected())

# 3. Set pump power
pump.verbose = True
pump.set_power(50)
print("Current power:", pump.get_power())

# 4. Activate suction
pump.activate_suction()
print("Is suction active?", pump.suction_active())

# 5. Change power again
pump.set_power(75)
print("Updated power:", pump.get_power())

# 6. Deactivate suction
pump.deactivate_suction()
print("Is suction active?", pump.suction_active())

# 7. Disconnect
pump.disconnect()
print("Is connected?", pump.is_connected())

# 8. Demonstrate error handling (uncomment to test)
# pump.set_power(30)   # should raise an exception since disconnected
# pump.get_power()     # should also raise an exception


Connected to test pipette pump
Is connected? True
Set power to 50%
Current power: 50
Activated suction
Is suction active? True
Set power to 75%
Updated power: 75
Deactivated suction
Is suction active? False
Disconnected from test pipette pump
Is connected? False


# Create and test a valve controller

In [4]:

vc = TestValveController()

# 1. Connect to the controller
vc.connect()

# 2. Check connection status
print("Is connected?", vc.is_connected())

# 3. Toggle some valves
vc.verbose = True
vc.toggle_valve(0, True)   # open valve 0
vc.toggle_valve(1, False)  # close valve 1
vc.toggle_valve(2, True)   # open valve 2

# 4. Get current valve states
print("Valve states:", vc.get_valve_states())

# 5. Change a valve state
vc.toggle_valve(1, True)   # open valve 1
print("Updated valve states:", vc.get_valve_states())

# 6. Disconnect and verify
vc.connected = False
print("Is connected?", vc.is_connected())

# 7. Demonstrate error handling (uncomment to test)
# vc.toggle_valve(3, True)    # should raise Exception since disconnected
# vc.get_valve_states()       # should also raise Exception since disconnected


Connected to test valve controller
Is connected? True
Set valve 0 to state True
Set valve 1 to state False
Set valve 2 to state True
Valve states: {0: True, 1: False, 2: True}
Set valve 1 to state True
Updated valve states: {0: True, 1: True, 2: True}
Is connected? False


# Demonstrate the controller widget next

In [ ]:
%gui qt6

from PyQt6.QtWidgets import QApplication

# 1) Create your back-end objects
mfc = TestMicrofluidicsController()   
valves = TestValveController()        
psu = TestPipettePump()               

# (Optional) If your pump object does NOT implement `disconnect_from_psu()`,
# wrap it so the widget's closeEvent doesn't error:
class _PSUAdapter:
    def __init__(self, pump): self.pump = pump
    def disconnect_from_psu(self): pass  # no-op for the test
# psu = _PSUAdapter(psu)

# 2) Shared control/state dict
n = mfc.get_number_channels()
c_p = {
    'current_pressures': [0.0]*n,
    'target_pressures':  [0.0]*n,
    'valves_used': [0, 1, 2],
    'valves_open': {0: False, 1: True, 2: False},
    'pipette_pump_current_power': 0.0,
    'pipette_pump_target_power':  3.3,
    'pipette_pump_on': True,
}

# 3) Launch the app + widget
app = QApplication.instance() or QApplication([])
w = MicrofluidicsControllerWidget(
    c_p,
    microfluidicsController=mfc,
    valve_controller=valves,
    pipette_pump=psu,
)
w.show()


Pump monitor started
